1. Environment Setup

This notebook prepares the environment for our Mini-BART research project.

In [ ]:
# %pip install -q "numpy==1.26.4" "transformers<5" datasets evaluate rouge_score sentencepiece accelerate

In [ ]:
# %pip install -q "scipy==1.15.3"

In [ ]:
# %pip install -q "numpy==2.2.6" "transformers==4.57.6" "datasets==3.6.0" evaluate rouge_score sentencepiece accelerate tensorboard pandas matplotlib pyyaml

2. Imports

Import the libraries required for this project.

In [ ]:
import numpy as np
import scipy
import torch
import transformers

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

3. GPU Verification

Verify that PyTorch can access our Colab GPU.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
import torch
import numpy as np
import scipy
import transformers
import datasets
import evaluate
import huggingface_hub
import fsspec

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

print("\nNumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("fsspec:", fsspec.__version__)
print("\nAll imports successful!")

4. Configuration

Define project-wide settings.

## 4. Loading the Pretrained BART Teacher

We use facebook/bart-large-cnn as our teacher model.

It is already pretrained and fine-tuned for CNN/DailyMail summarization.

Our objective is to use this teacher to guide the training of a smaller BART student through knowledge distillation.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

teacher = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

teacher = teacher.to("cuda")
teacher.eval()

print("Teacher loaded successfully!")
print("Model:", MODEL_NAME)
print("Parameters:", sum(p.numel() for p in teacher.parameters()))

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

# Create a permanent storage location
TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
os.makedirs(TEACHER_DIR, exist_ok=True)

# Save the model and tokenizer
teacher.save_pretrained(TEACHER_DIR)
tokenizer.save_pretrained(TEACHER_DIR)

print("Teacher model and tokenizer saved successfully!")
print("Location:", TEACHER_DIR)


In [ ]:
print(os.listdir(TEACHER_DIR))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

teacher_path = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

test_tokenizer = AutoTokenizer.from_pretrained(teacher_path)
test_model = AutoModelForSeq2SeqLM.from_pretrained(teacher_path)

print("Reload successful!")
print("Parameters:", sum(p.numel() for p in test_model.parameters()))

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

print("Teacher directory exists:", os.path.exists(TEACHER_DIR))
print("Files:", os.listdir(TEACHER_DIR))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)

teacher = AutoModelForSeq2SeqLM.from_pretrained(
    TEACHER_DIR
).to("cuda")

teacher.eval()

print("Teacher loaded successfully!")
print("Parameters:", sum(p.numel() for p in teacher.parameters()))
print("Device:", next(teacher.parameters()).device)

In [ ]:
article = """
The Indian Space Research Organisation (ISRO) successfully launched
its latest Earth observation satellite aboard a Polar Satellite Launch
Vehicle (PSLV) from the Satish Dhawan Space Centre in Sriharikota.
The satellite will provide high-resolution images to support
agriculture, disaster management, urban planning, and environmental
monitoring. Scientists stated that the mission demonstrates India's
continued progress in developing reliable space technology.
"""

inputs = tokenizer(
    article,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to("cuda")

with torch.no_grad():
    summary_ids = teacher.generate(
        **inputs,
        max_length=100,
        num_beams=4,
        forced_bos_token_id=0
    )

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Teacher summary:\n", summary)

In [ ]:

from datasets import load_dataset

# Load 5 examples from the test split
dataset = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split="test[:5]"
)

print("Dataset loaded successfully!")
print("Number of examples:", len(dataset))
print("Columns:", dataset.column_names)


In [ ]:
import datasets
import huggingface_hub

print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0",
    split="test[:5]"
)

print("Dataset loaded successfully!")
print("Number of examples:", len(dataset))
print("Columns:", dataset.column_names)

In [ ]:
sample = dataset[0]

print("ARTICLE:\n")
print(sample["article"][:1500])

print("\nREFERENCE SUMMARY:\n")
print(sample["highlights"])

In [ ]:

import torch
teacher.eval()
teacher_results = []
for i, example in enumerate(dataset):

    article = example["article"]
    reference = example["highlights"]

    # Tokenize the article
    inputs = tokenizer(
        article,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to("cuda")

    # Generate summary
    with torch.no_grad():
        summary_ids = teacher.generate(
            **inputs,
            max_length=128,
            min_length=30,
            num_beams=4,
            forced_bos_token_id=0
        )

    # Convert generated tokens into text
    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    teacher_results.append({
        "id": example["id"],
        "article": article,
        "reference_summary": reference,
        "teacher_summary": generated_summary
    })

    print(f"Completed article {i + 1}/5")

print("\nAll teacher summaries generated!")


In [ ]:
for i, result in enumerate(teacher_results):

    print(f"\n{'='*80}")
    print(f"ARTICLE {i+1}")
    print(f"{'='*80}")

    print("\nREFERENCE SUMMARY:\n")
    print(result["reference_summary"])

    print("\nTEACHER SUMMARY:\n")
    print(result["teacher_summary"])

In [ ]:
import json
import os

RESULTS_DIR = (
    "/content/drive/MyDrive/research_artifacts/mini_bart/"
    "teacher_baseline"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

output_path = os.path.join(
    RESULTS_DIR,
    "teacher_baseline_5_samples.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        teacher_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved to:", output_path)

In [ ]:
for i, result in enumerate(teacher_results):
    print(f"\nArticle {i+1}")
    print("ID:", result["id"])
    print("Article length:", len(result["article"]))
    print("Reference:", result["reference_summary"][:150])
    print("Teacher:", result["teacher_summary"])

In [ ]:
print("Total results:", len(teacher_results))

print("\nArticle 5")
print("ID:", teacher_results[4]["id"])
print("Article length:", len(teacher_results[4]["article"]))

print("\nReference summary:")
print(teacher_results[4]["reference_summary"])

print("\nTeacher summary:")
print(teacher_results[4]["teacher_summary"])

In [ ]:
%pip install -q evaluate rouge_score

In [ ]:
import evaluate
rouge = evaluate.load("rouge")
print("ROUGE metric loaded successfully!")

In [ ]:
predictions = [
    result["teacher_summary"]
    for result in teacher_results
]

references = [
    result["reference_summary"]
    for result in teacher_results
]

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("Teacher ROUGE Scores:\n")

for metric, score in rouge_scores.items():
    print(f"{metric}: {score:.4f}")

MEASURING EXTRACTIVENESS (E0)

Unigram overlap: copied individual words

Bigram overlap: copied consecutive two-word phrases
 
4-gram overlap: copied consecutive four-word phrases

copy rate= summary n-grams found in source/total summary n grams

Novel rate = 1-copy rate

In [ ]:
import re
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())
def get_ngrams(tokens, n):
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]
def source_overlap(source, summary, n):
    source_tokens = tokenize(source)
    summary_tokens = tokenize(summary)

    source_ngrams = set(get_ngrams(source_tokens, n))
    summary_ngrams = get_ngrams(summary_tokens, n)

    if not summary_ngrams:
        return 0.0

    copied = sum(
        ngram in source_ngrams
        for ngram in summary_ngrams
    )

    return copied / len(summary_ngrams)

In [ ]:
copy_rates = {1: [], 2: [], 4: []}
for result in teacher_results:

    # Reconstruct the exact truncated article input
    encoded = tokenizer(
        result["article"],
        max_length=1024,
        truncation=True
    )

    source_text = tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=True
    )

    summary = result["teacher_summary"]

    for n in [1, 2, 4]:
        rate = source_overlap(source_text, summary, n)
        copy_rates[n].append(rate)


print("Teacher Source Overlap:\n")

for n in [1, 2, 4]:
    avg_copy = sum(copy_rates[n]) / len(copy_rates[n])
    avg_novel = 1 - avg_copy

    print(f"{n}-gram Copy Rate:  {avg_copy:.4f}")
    print(f"{n}-gram Novel Rate: {avg_novel:.4f}\n")

SAVING BASELINE RESULT

will compare it with E1 AND E2 LATER

SAVING ROUGE METRICS and EXTRACTIVENESS RECORD IN GOOGLE DRIVE 

In [ ]:
import json
import os
baseline_metrics = {
    "experiment": "E0_teacher",
    "num_examples": len(teacher_results),
    "rouge": rouge_scores,
    "source_overlap": {
        f"{n}-gram": {
            "copy_rate": sum(copy_rates[n]) / len(copy_rates[n]),
            "novel_rate": 1 - sum(copy_rates[n]) / len(copy_rates[n])
        }
        for n in [1, 2, 4]
    }
}

RESULTS_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher_baseline"
os.makedirs(RESULTS_DIR, exist_ok=True)

output_path = os.path.join(RESULTS_DIR, "teacher_metrics.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=4)

print("Teacher baseline metrics saved!")
print(output_path)

In [ ]:
import os

print("Current working directory:", os.getcwd())

print("\nChecking project directory:")

for path in [
    "projects/01-mini-bart/src/metrics.py",
    "/content/Research/projects/01-mini-bart/src/metrics.py"
]:
    print(path, "->", os.path.exists(path))

In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive")[:20])

In [ ]:
import os
research_path = "/content/drive/MyDrive/Research"

print("Research folder exists:", os.path.exists(research_path))

if os.path.exists(research_path):
    print("\nContents:")
    print(os.listdir(research_path))

VS Code (Windows)
Source of truth, Write and edit metrics.py, notebooks and other project files.

Google Drive
Make the repository files accessible to Colab.

Colab T4
Import and execute the Python modules remotely.

cloning REPO IN DRIVE

In [ ]:
#cloning repo in drive
%cd /content/drive/MyDrive

!git clone https://github.com/HeavenlyDemon27/Research.git

verifying the cloning

In [ ]:
import os
PROJECT_ROOT = "/content/drive/MyDrive/Research"

print("Repository exists:", os.path.exists(PROJECT_ROOT))
print("Project files:", os.listdir(PROJECT_ROOT))

Sync the latest code to the remote runtime

This updates the Drive copy with the metrics.py I just pushed.

In [ ]:
%cd /content/drive/MyDrive/Research
!git pull --ff-only origin main

Import our functions

We're importing our own reusable functions rather than keeping all evaluation logic inside the noteboo

In [ ]:
import sys
SRC_DIR = "/content/drive/MyDrive/Research/projects/01-mini-bart/src"

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from metrics import compute_rouge, compute_source_overlap

print("Metrics module imported successfully!")

Testing module against our teacher results

In [ ]:
predictions = [r["teacher_summary"] for r in teacher_results]
references = [r["reference_summary"] for r in teacher_results]
sources = []
for r in teacher_results:
    encoded = tokenizer(
        r["article"],
        max_length=1024,
        truncation=True
    )

    source = tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=True
    )

    sources.append(source)

# Calculate metrics using our reusable module
rouge_scores = compute_rouge(predictions, references)

overlap_scores = compute_source_overlap(
    sources,
    predictions
)

print("ROUGE:", rouge_scores)
print("\nSource overlap:", overlap_scores)

ABOVE there, What are we checking?

We're checking whether the functions in metrics.py reproduce the evaluation results we previously calculated directly in the notebook.

Expected ROUGE results should be approximately:

Metric	Previous result

ROUGE-1	0.4674

ROUGE-2	0.2629

ROUGE-L	0.3522

ROUGE-Lsum	0.4082

Small differences may arise from implementation or rounding, but the results should match if the same inputs and settings are used

FINISH E0- SAVING FINAL TEACHERS METRICS

In [ ]:
import json
import os
baseline_metrics = {
    "experiment": "E0_teacher",
    "num_examples": len(teacher_results),

    "rouge": {
        key: float(value)
        for key, value in rouge_scores.items()
    },

    "source_overlap": overlap_scores
}

RESULTS_DIR = (
    "/content/drive/MyDrive/research_artifacts/"
    "mini_bart/teacher_baseline"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

output_path = os.path.join(
    RESULTS_DIR,
    "teacher_metrics.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=4)

print("Teacher metrics saved!")
print(output_path)

VERIFY

In [ ]:
#verifying
print(os.listdir(RESULTS_DIR))

E0 EVALUATION COMPLETED. YAAAAYYYYY🥳🥳